In [ ]:
%pip -q install numpy pandas scikit-learn joblib torch optuna

In [ ]:
from pathlib import Path
import sys
import importlib
from urllib.request import urlopen

NOTEBOOK_DIR = Path.cwd() / "KC_CO2_Modular_Notebooks"
if not (NOTEBOOK_DIR / "kc_common.py").is_file():
    NOTEBOOK_DIR = Path.cwd()
REPO_REF = "ad2d2f6dbb61a8e06264703db43272863f5ce124"
helper_path = NOTEBOOK_DIR / "kc_common.py"
if not helper_path.is_file():
    helper_url = f"https://raw.githubusercontent.com/Namelessr679/ESP32-Indoor-Air-Quality-1.0Monitor/{REPO_REF}/KC_CO2_Modular_Notebooks/kc_common.py"
    with urlopen(helper_url, timeout=30) as response:
        helper_source = response.read()
    compile(helper_source, "kc_common.py", "exec")
    if b"def prepare_experiment(" not in helper_source:
        raise RuntimeError("The downloaded helper is incompatible with this notebook revision.")
    NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
    helper_path.write_bytes(helper_source)
module_dir = NOTEBOOK_DIR / "models"
module_dir.mkdir(parents=True, exist_ok=True)
for module_name in ("gru_pipeline.py",):
    module_path = module_dir / module_name
    if not module_path.is_file():
        module_url = f"https://raw.githubusercontent.com/Namelessr679/ESP32-Indoor-Air-Quality-1.0Monitor/{REPO_REF}/KC_CO2_Modular_Notebooks/models/{module_name}"
        with urlopen(module_url, timeout=30) as response:
            module_path.write_bytes(response.read())
sys.path.insert(0, str(NOTEBOOK_DIR))
import kc_common
if Path(kc_common.__file__).resolve() != helper_path.resolve():
    raise RuntimeError("A different kc_common module is already loaded. Restart the runtime and rerun setup.")
importlib.reload(kc_common)
from kc_common import experiment_run_dir, load_prepared


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/KC_CO2_Modular_Notebooks")
RUN_DIR = experiment_run_dir(PROJECT)


In [ ]:
prepared = load_prepared(RUN_DIR)

MODEL_NAME = 'GRU'
configurations = prepared["configurations"]
history_by_session = prepared["history_by_session"]
OUT = RUN_DIR / MODEL_NAME


In [ ]:
from IPython.display import display
import sys

from models.gru_pipeline import run_direct, run_recursive

gru_direct_results = run_direct("GRU", RUN_DIR, RUN_DIR / "GRU")
gru_recursive_results = run_recursive("GRU", RUN_DIR, RUN_DIR / "GRU")
display(gru_direct_results["metrics"])
display(gru_recursive_results["metrics"])
print("Saved GRU artifacts:", RUN_DIR / "GRU")